# 📧 BERT-Based Email Spam Classifier
## Using Sentence-Transformers (all-MiniLM-L6-v2) for Semantic Spam Detection

---
**Model   :** `sentence-transformers/all-MiniLM-L6-v2` (384-dim BERT embeddings)  
**Dataset :** UCI SMS Spam Collection (~5 572 labelled messages)  
**Pipeline:** Text → BERT Embedding → Classifier (LR / SVM / MLP)

### Table of Contents
1. [Setup & Imports](#1)
2. [Data Loading & EDA](#2)
3. [BERT Embeddings](#3)
4. [t-SNE Visualisation](#4)
5. [Model Training](#5)
6. [Evaluation](#6)
7. [Save Best Model](#7)
8. [Interactive Prediction](#8)

## 1. Setup & Imports <a id='1'></a>

In [ ]:
# Uncomment to install on first run:
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'])

import os, re, io, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay,
)

BASE_DIR   = os.getcwd()
DATA_DIR   = os.path.join(BASE_DIR, 'data')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
for d in [DATA_DIR, MODEL_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Setup complete!')

## 2. Data Loading & EDA <a id='2'></a>

In [ ]:
DATASET_URL = (
    'https://archive.ics.uci.edu/ml/machine-learning-databases/'
    '00228/smsspamcollection.zip'
)
CSV_PATH = os.path.join(DATA_DIR, 'spam.csv')

def download_dataset():
    print('⬇  Downloading SMS Spam Collection …')
    with urllib.request.urlopen(DATASET_URL) as resp:
        raw = resp.read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        with z.open('SMSSpamCollection') as f:
            content = f.read().decode('utf-8')
    rows = [line.split('\t', 1) for line in content.strip().splitlines()]
    df = pd.DataFrame(rows, columns=['label', 'text'])
    df.to_csv(CSV_PATH, index=False)
    return df

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    if 'label' not in df.columns:
        df = download_dataset()
else:
    df = download_dataset()

df = df[['label', 'text']].dropna()
df['label'] = df['label'].str.strip()
print(f'Loaded {len(df)} messages')
df.head(8)

In [ ]:
# ── Class distribution + message length ──────────────────────────────────────
df['msg_len'] = df['text'].str.len()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Pie
counts = df['label'].value_counts()
axes[0].pie(counts, labels=['Ham', 'Spam'], autopct='%1.1f%%',
            colors=['#4ECDC4', '#FF6B6B'], startangle=90,
            explode=[0, 0.05], shadow=True, textprops={'fontsize': 13})
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')

# Bar
bars = axes[1].bar(counts.index, counts.values, color=['#4ECDC4', '#FF6B6B'],
                   edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 str(v), ha='center', fontsize=13, fontweight='bold')
axes[1].set_title('Message Count', fontsize=13, fontweight='bold')

# Length histogram
for label, color in [('ham', '#4ECDC4'), ('spam', '#FF6B6B')]:
    axes[2].hist(df.loc[df['label']==label, 'msg_len'],
                 bins=40, alpha=0.65, color=color, label=label.capitalize(), edgecolor='white')
axes[2].set_title('Message Length Distribution', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Characters'); axes[2].legend()

plt.suptitle('SMS Spam Collection – EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'eda.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Word clouds ───────────────────────────────────────────────────────────────
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, label, cmap, title in [
        (axes[0], 'ham',  'cool', 'HAM – Word Cloud'),
        (axes[1], 'spam', 'Reds', 'SPAM – Word Cloud'),
    ]:
        blob = ' '.join(df.loc[df['label']==label, 'text'])
        wc = WordCloud(width=700, height=380, background_color='white',
                       colormap=cmap, max_words=150).generate(blob)
        ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
        ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'wordclouds.png'), dpi=120, bbox_inches='tight')
    plt.show()
except ImportError:
    print('wordcloud not installed – pip install wordcloud')

## 3. BERT Embeddings <a id='3'></a>

In [ ]:
# ── Why BERT embeddings? ──────────────────────────────────────────────────────
#
#  TF-IDF:  Represents words as independent counts. No semantics.
#           "win prize" ≠ "claim reward" (different vectors)
#
#  BERT:    Encodes meaning. "win prize" ≈ "claim reward" (similar vectors)
#           Handles paraphrasing, synonyms, context.
#
#  Model:   all-MiniLM-L6-v2
#           • 6-layer MiniLM distilled from BERT-large
#           • 384-dimensional sentence vectors
#           • Trained on 1B+ sentence pairs
#           • ~80 MB | fast on CPU

from sentence_transformers import SentenceTransformer

MODEL_NAME = 'all-MiniLM-L6-v2'

def preprocess(text):
    """Minimal clean — BERT handles tokenisation internally."""
    text = re.sub(r'http\S+|www\S+', 'URL', text)
    return text.strip()

df['text_clean'] = df['text'].apply(preprocess)

print(f'Loading {MODEL_NAME} … (downloads ~80 MB on first run)')
encoder = SentenceTransformer(MODEL_NAME)

print('Encoding messages …')
X_emb = encoder.encode(
    df['text_clean'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f'\n✅ Embedding matrix shape: {X_emb.shape}')  # (5572, 384)
print(f'   Each row = 384-dim semantic vector for one message')

In [ ]:
# ── Cosine similarity: spam vs. ham examples ──────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

samples = {
    'spam_1': 'Congratulations! You won a FREE iPhone. Click now!',
    'spam_2': 'URGENT: Claim your $5000 cash prize immediately!',
    'ham_1' : 'Hey, are we still meeting for lunch tomorrow?',
    'ham_2' : 'Can you send me the lecture notes please?',
}
sample_embs = encoder.encode(list(samples.values()), convert_to_numpy=True)
sim = cosine_similarity(sample_embs)

df_sim = pd.DataFrame(sim, index=samples.keys(), columns=samples.keys())
plt.figure(figsize=(7, 5))
sns.heatmap(df_sim, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, annot_kws={'size': 11})
plt.title('Cosine Similarity between BERT Embeddings', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'bert_cosine_sim.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Notice: spam messages cluster together; ham messages cluster together.')

## 4. t-SNE Visualisation of BERT Embeddings <a id='4'></a>

In [ ]:
from sklearn.manifold import TSNE

le = LabelEncoder()
y  = le.fit_transform(df['label'])   # ham=0, spam=1
LABEL_NAMES = le.classes_.tolist()

# Subsample for speed (t-SNE is O(n²))
N = 2000
idx = np.random.default_rng(42).choice(len(X_emb), N, replace=False)
X_sub, y_sub = X_emb[idx], y[idx]

print(f'Running t-SNE on {N} samples (perplexity=40) …')
tsne = TSNE(n_components=2, perplexity=40, random_state=42, max_iter=1000)
X2   = tsne.fit_transform(X_sub)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#4ECDC4', '#FF6B6B']
for cls, (label, color) in enumerate(zip(LABEL_NAMES, colors)):
    mask = y_sub == cls
    ax.scatter(X2[mask, 0], X2[mask, 1], c=color, label=label.capitalize(),
               alpha=0.55, s=20, edgecolors='none')

ax.set_title('t-SNE of BERT Embeddings\n(spam vs. ham clusters)', fontsize=14, fontweight='bold')
ax.legend(fontsize=13, markerscale=2)
ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'bert_tsne.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Clear cluster separation → BERT embeddings are highly discriminative.')

## 5. Model Training <a id='5'></a>

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_emb, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_tr)}   Test: {len(X_te)}')

MODELS = {
    'Logistic Regression': LogisticRegression(C=5, max_iter=1000, random_state=42),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(C=1, max_iter=3000, random_state=42)),
    'MLP Neural Network': MLPClassifier(
        hidden_layer_sizes=(256, 128), activation='relu',
        max_iter=300, random_state=42,
        early_stopping=True, validation_fraction=0.1,
    ),
}

def evaluate_model(name, clf, X_tr, X_te, y_tr, y_te):
    clf.fit(X_tr, y_tr)
    y_pred  = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)[:,1] if hasattr(clf,'predict_proba') else None
    return dict(
        name=name, clf=clf, y_pred=y_pred, y_proba=y_proba,
        accuracy =accuracy_score(y_te, y_pred),
        precision=precision_score(y_te, y_pred, pos_label=1),
        recall   =recall_score(y_te, y_pred, pos_label=1),
        f1       =f1_score(y_te, y_pred, pos_label=1),
        roc_auc  =roc_auc_score(y_te, y_proba) if y_proba is not None else float('nan'),
    )

print('\n🏋  Training classifiers on BERT embeddings …\n')
results = []
for name, clf in MODELS.items():
    r = evaluate_model(name, clf, X_tr, X_te, y_tr, y_te)
    results.append(r)
    print(f'  ✓ {name:<25}  F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}')

## 6. Evaluation <a id='6'></a>

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
df_res = pd.DataFrame([{
    'Model': r['name'], 'Accuracy': r['accuracy'],
    'Precision': r['precision'], 'Recall': r['recall'],
    'F1': r['f1'], 'ROC-AUC': r['roc_auc'],
} for r in results])

display(df_res.style
    .format('{:.4f}', subset=['Accuracy','Precision','Recall','F1','ROC-AUC'])
    .highlight_max(subset=['Accuracy','Precision','Recall','F1','ROC-AUC'], color='#d4f5d4')
    .set_properties(**{'text-align':'center'}))

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, res in zip(axes, results):
    cm = confusion_matrix(y_te, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
                ax=ax, annot_kws={'size': 15})
    ax.set_title(res['name'], fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices – BERT Classifiers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'bert_confusion_matrices.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.Set1.colors
for i, res in enumerate(results):
    if res['y_proba'] is not None:
        RocCurveDisplay.from_predictions(
            y_te, res['y_proba'],
            name=f"{res['name']} (AUC={res['roc_auc']:.4f})",
            ax=ax, color=colors[i],
        )
ax.plot([0,1],[0,1],'k--',lw=1.2)
ax.set_title('ROC Curves – BERT Classifiers', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'bert_roc_curves.png'), dpi=120)
plt.show()

In [ ]:
# ── Detailed report for best model ────────────────────────────────────────────
best = max(results, key=lambda r: r['f1'])
print(f'🏆 Best model: {best["name"]}  (F1={best["f1"]:.4f})')
print()
print(classification_report(y_te, best['y_pred'], target_names=LABEL_NAMES))

## 7. Save Best Model <a id='7'></a>

In [ ]:
joblib.dump(best['clf'], os.path.join(MODEL_DIR, 'bert_best_model.pkl'))
joblib.dump(encoder,    os.path.join(MODEL_DIR, 'bert_encoder.pkl'))
joblib.dump(le,         os.path.join(MODEL_DIR, 'label_encoder.pkl'))

print('💾 Saved:')
for f in os.listdir(MODEL_DIR):
    size = os.path.getsize(os.path.join(MODEL_DIR, f)) / 1024
    print(f'   models/{f}  ({size:.0f} KB)')

## 8. Interactive Prediction <a id='8'></a>

In [ ]:
clf_loaded     = joblib.load(os.path.join(MODEL_DIR, 'bert_best_model.pkl'))
encoder_loaded = joblib.load(os.path.join(MODEL_DIR, 'bert_encoder.pkl'))
le_loaded      = joblib.load(os.path.join(MODEL_DIR, 'label_encoder.pkl'))

def classify(text: str):
    clean   = re.sub(r'http\S+|www\S+', 'URL', text).strip()
    X       = encoder_loaded.encode([clean], convert_to_numpy=True)
    label   = le_loaded.inverse_transform(clf_loaded.predict(X))[0]
    proba   = clf_loaded.predict_proba(X)[0] if hasattr(clf_loaded, 'predict_proba') else None
    spam_idx = list(le_loaded.classes_).index('spam')
    conf    = proba[spam_idx] if proba is not None else None
    icon    = '🚨 SPAM' if label == 'spam' else '✅  HAM'
    conf_s  = f'  ({conf*100:.1f}% spam probability)' if conf is not None else ''
    print(f'Message : {text[:100]}')
    print(f'Result  : {icon}{conf_s}')
    print()

test_messages = [
    'Congratulations! You have won a FREE iPhone. Click now to claim your prize!',
    'Hey, are we still meeting for lunch tomorrow at noon?',
    'URGENT: Your bank account has been suspended. Call 0800-FREE-CASH immediately.',
    'Can you please send me the slides from today\'s lecture?',
    'You have been selected for a SPECIAL cash reward. Reply YES to claim.',
    'The project deadline has been extended to next Friday.',
    'FREE ringtones! txt RING to 87070. 3 for FREE! 16+ TCs apply.',
    'Mom is calling, please call her back when you have a moment.',
]

print('='*65)
print('  📧  BERT Spam Classifier – Prediction Examples')
print('='*65 + '\n')
for msg in test_messages:
    classify(msg)